In [ ]:
import pandas as pd
import numpy as np
import gdown
import re
import os
from pathlib import Path

folder_url = "https://drive.google.com/drive/folders/1WMeIajroEakGEvWeIl75HM6DxTc1GJDv?usp=drive_link"
base_dir = Path("/content/temp_data")
base_dir.mkdir(parents=True, exist_ok=True)

match = re.search(r'folders/([\w-]+)', folder_url)
if match:
    folder_id = match.group(1)
    gdown.download_folder(id=folder_id, output=str(base_dir), quiet=True, use_cookies=False)

    subfolders = [f.path for f in os.scandir(base_dir) if f.is_dir()]
    DATA_DIR = (subfolders[0] + '/') if subfolders else (str(base_dir) + '/')
else:
    print("Lỗi")

orders      = pd.read_csv(DATA_DIR + 'orders.csv',      parse_dates=['order_date'])
products    = pd.read_csv(DATA_DIR + 'products.csv')
returns     = pd.read_csv(DATA_DIR + 'returns.csv')
web_traffic = pd.read_csv(DATA_DIR + 'web_traffic.csv', parse_dates=['date'])
order_items = pd.read_csv(DATA_DIR + 'order_items.csv', low_memory=False)
customers   = pd.read_csv(DATA_DIR + 'customers.csv')
payments    = pd.read_csv(DATA_DIR + 'payments.csv')
geography   = pd.read_csv(DATA_DIR + 'geography.csv')


Q1. Trung vị số ngày giữa hai lần mua liên tiếp (inter-order gap)

In [ ]:
# 1. Sap xep
orders_sorted = orders.sort_values(['customer_id', 'order_date'])

# 2. Tinh gap
gaps = orders_sorted.groupby('customer_id')['order_date'].diff().dt.days.dropna()

# 3. Median
print(f"Median: {gaps.median()} ngay")

Median: 144.0 ngay


Q2. Phân khúc sản phẩm có tỷ suất lợi nhuận gộp trung bình cao nhất

In [ ]:
print(f'cogs >= price: {(products["cogs"] >= products["price"]).sum()} dong')

# gross margin = (price - cogs) / price
products['gross_margin'] = (products['price'] - products['cogs']) / products['price']

gm_by_segment = products.groupby('segment')['gross_margin'].mean().sort_values(ascending=False)
print(gm_by_segment.to_string())

gm_by_segment.idxmax()

cogs >= price: 0 dong
segment
Standard       0.313442
Premium        0.285377
All-weather    0.284176
Activewear     0.265600
Performance    0.263650
Balanced       0.258038
Trendy         0.240758
Everyday       0.236343


'Standard'

Q3. Lý do trả hàng phổ biến nhất trong danh mục Streetwear

In [ ]:
# Lọc sản phẩm Streetwear
streetwear_ids = products[products['category'] == 'Streetwear'][['product_id']]

# Join
ret_streetwear = returns.merge(streetwear_ids, on='product_id')

reason_counts = ret_streetwear['return_reason'].value_counts()
print(reason_counts.to_string())

return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159


Q4. Nguồn truy cập có tỷ lệ thoát (bounce_rate) trung bình thấp nhất

In [ ]:
avg_bounce = web_traffic.groupby('traffic_source')['bounce_rate'].mean().sort_values()
print('Tỷ lệ bounce trung bình theo nguồn traffic:')
print(avg_bounce.to_string())
print(f'\n{avg_bounce.idxmin()}')

Tỷ lệ bounce trung bình theo nguồn traffic:
traffic_source
email_campaign    0.004458
social_media      0.004476
paid_search       0.004478
referral          0.004499
organic_search    0.004504
direct            0.004511

email_campaign


Q5. Tỷ lệ % dòng order_items có áp dụng khuyến mãi (promo_id không null)

In [ ]:
pct_promo = order_items['promo_id'].notna().mean() * 100
print(f'Tỷ lệ order_items có promo_id: {pct_promo:.2f}%')

Tỷ lệ order_items có promo_id: 38.66%


Q6. Nhóm tuổi có số đơn hàng trung bình trên mỗi khách hàng cao nhất

In [ ]:
# Số đơn mỗi KH
orders_per_cust = orders.groupby('customer_id').size().reset_index(name='order_count')

# Chỉ lấy khách có age_group không null
cust_age = customers[customers['age_group'].notna()].merge(
    orders_per_cust, on='customer_id', how='left'
).fillna(0)

avg_orders = cust_age.groupby('age_group')['order_count'].mean().sort_values(ascending=False)
print(avg_orders.to_string())
print(f'\n{avg_orders.idxmax()}')

age_group
55+      5.406851
45-54    5.357241
35-44    5.337343
25-34    5.245226
18-24    5.226656

55+


Q7. Vùng tạo ra tổng doanh thu cao nhất

In [ ]:
# 1. Revenue: unit_price đã là giá sau giảm
order_items['line_revenue'] = order_items['quantity'] * order_items['unit_price']
order_rev = order_items.groupby('order_id')['line_revenue'].sum().reset_index()

# 2. Lọc đơn hàng hợp lệ theo status
valid_statuses = ['delivered', 'shipped', 'paid', 'returned']
orders_valid = orders[orders['order_status'].isin(valid_statuses)][['order_id', 'zip']]

# 3. Join
order_geo = (
    orders_valid
    .merge(order_rev, on='order_id')
    .merge(geography[['zip', 'region']], on='zip')
)

# 4. Tổng hợp theo vùng
rev_by_region = order_geo.groupby('region')['line_revenue'].sum().sort_values(ascending=False)

print('Tổng doanh thu thuần theo vùng:')
print(rev_by_region.apply(lambda x: f'{x:,.0f}').to_string())
print(f'\n{rev_by_region.idxmax()}')

Tổng doanh thu thuần theo vùng:
region
East       6,841,337,922
Central    4,423,476,255
West       3,459,119,071

East


In [ ]:
import pandas as pd

# Lọc hóa đơn trong đúng ngày 2012-07-04
orders_test = orders[orders['order_date'].dt.strftime('%Y-%m-%d') == '2012-07-04']

# Join với order_items
df_test = orders_test.merge(order_items, on='order_id')

df_test['revenue'] = df_test['quantity'] * df_test['unit_price'] - df_test['discount_amount']

breakdown = df_test.groupby('order_status')['revenue'].sum()

print("DOANH THU NGÀY 2012-07-04:")
for status, rev in breakdown.items():
    print(f"{status:<15}: {rev:,.2f}")

print(f"Tổng tất cả status       : {breakdown.sum():,.2f}")
print(f"Ghi nhận trong sales.csv : 5,123,547.94")


DOANH THU NGÀY 2012-07-04:
cancelled      : 154,785.82
delivered      : 4,523,077.12
paid           : 75,164.62
returned       : 318,628.14
shipped        : 51,892.24
Tổng tất cả status       : 5,123,547.94
Ghi nhận trong sales.csv : 5,123,547.94


Q8. Phương thức thanh toán phổ biến nhất trong đơn bị huỷ (cancelled)

In [ ]:
# Lọc đơn cancelled
cancelled = orders[orders['order_status'] == 'cancelled']

payment_counts = cancelled['payment_method'].value_counts()
print('Phương thức thanh toán trong đơn cancelled:')
print(payment_counts.to_string())
print(f'\n{payment_counts.idxmax()}')

Phương thức thanh toán trong đơn cancelled:
payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535

credit_card


Q9. Kích thước sản phẩm có tỷ lệ trả hàng cao nhất

In [ ]:
# Join
oi_with_size  = order_items.merge(products[['product_id', 'size']], on='product_id')
ret_with_size = returns.merge(products[['product_id', 'size']], on='product_id')

sizes = ['S', 'M', 'L', 'XL']
oi_size  = oi_with_size[oi_with_size['size'].isin(sizes)].groupby('size').size()
ret_size = ret_with_size[ret_with_size['size'].isin(sizes)].groupby('size').size()

return_rate = (ret_size / oi_size).sort_values(ascending=False)
print(return_rate.apply(lambda x: f'{x:.4f} ({x*100:.2f}%)').to_string())
print(f'\n{return_rate.idxmax()}')

size
S     0.0565 (5.65%)
L     0.0562 (5.62%)
M     0.0557 (5.57%)
XL    0.0552 (5.52%)

S


Q10. Kế hoạch trả góp có giá trị thanh toán trung bình cao nhất


In [ ]:
avg_payment = payments.groupby('installments')['payment_value'].mean().sort_values(ascending=False)
print('Giá trị thanh toán trung bình theo số kỳ trả góp:')
print(avg_payment.apply(lambda x: f'{x:,.2f}').to_string())
print(f'\n{avg_payment.idxmax()}')

Giá trị thanh toán trung bình theo số kỳ trả góp:
installments
6     24,446.65
3     24,399.64
12    24,245.77
1     24,113.27
2        708.47

6
